## 0. Fluent 실행 by pyfluent

In [ ]:
# ============================================================
# [설정 파라미터] - 모든 변수는 이 셀에서 관리합니다
# ============================================================

# --- Case / 출력 파일 경로 ---
case_file  = r"C:/02_PEMFC3_5cm2/validation/5cm2_ref_5V.cas.h5"
output_dir = r"C:/02_PEMFC3_5cm2/validation"

# --- Anode 조건 ---
T_an  = 80        # 온도 [°C]
RH_an = 100       # 상대습도 [%]
P_an  = 101325    # 압력 [Pa]

# --- Cathode 조건 ---
T_ca  = 80        # 온도 [°C]
RH_ca = 100       # 상대습도 [%]
P_ca  = 101325    # 압력 [Pa]

# --- 반복 계산 횟수 ---
iter_init = 100   # 초기화 후 반복 횟수 (단순유동)
iter_second = 10000000 # 초기화 후 반복 횟수 (전기화학반응)

# operation Voltage
oper_volt = 0.4

# --- IV 커브 전압 범위 [V] ---
voltage_range = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

# --- Mesh 변경용 (섹션 14) ---
mesh_file          = r"C:/02_PEMFC3_5cm2/mesh_independent/5cm2_ref_40mesh.msh"
membrane_material  = "nafion-nr211"

In [130]:
import ansys.fluent.core as pyfluent
solver = pyfluent.launch_fluent(precision='double', 
                                processor_count=8, 
                                product_version='25.2.0', 
                                mode='solver', 
                                ui_mode = 'gui'
                                )

## - Case File Reader
1. case 파일 read
2. UDF unload
3. UDF load
4. Execute on Demand for User-Defined variables
5. case 파일 read again

- 코드 변수: "file_name"

In [ ]:
solver.settings.file.read_case(file_name=case_file)
solver.tui.define.user_defined.compiled_functions("unload", "libudf")
solver.tui.define.user_defined.compiled_functions("load", "libudf")
solver.tui.define.user_defined.execute_on_demand('"rename_UDvars::libudf"')
solver.settings.file.read_case(file_name=case_file)

##### 계산식

In [ ]:
import numpy as np

# ============================================================
# Molecular Weights [g/mol]
# ============================================================
M_H2  = 2.016
M_O2  = 32.000
M_N2  = 28.014
M_H2O = 18.015

# ============================================================
# Saturation Pressure (Antoine Equation)
#   log10(P_sat [mmHg]) = 8.10765 - 1750.286 / (235.0 + T [°C])
# ============================================================
def sat_pressure(T_C):
    """포화수증기압 계산 [Pa], T: 온도 [°C]"""
    log_P_mmHg = 8.10765 - 1750.286 / (235.0 + T_C)
    return (10 ** log_P_mmHg) * 133.322   # mmHg → Pa

# ============================================================
# Anode: H2 + H2O
# ============================================================
P_sat_an  = sat_pressure(T_an)
P_H2O_an  = (RH_an / 100) * P_sat_an
P_H2_an   = P_an - P_H2O_an

x_H2_an   = P_H2_an  / P_an
x_H2O_an  = P_H2O_an / P_an

M_mix_an  = x_H2_an * M_H2 + x_H2O_an * M_H2O

anode_Y_h2  = (x_H2_an  * M_H2)  / M_mix_an
anode_Y_h2o = (x_H2O_an * M_H2O) / M_mix_an

# ============================================================
# Cathode: O2 + N2 + H2O  (건조공기: O2 21%, N2 79%)
# ============================================================
P_sat_ca      = sat_pressure(T_ca)
P_H2O_ca      = (RH_ca / 100) * P_sat_ca
P_dry_air_ca  = P_ca - P_H2O_ca
P_O2_ca       = 0.21 * P_dry_air_ca
P_N2_ca       = 0.79 * P_dry_air_ca

x_O2_ca   = P_O2_ca  / P_ca
x_N2_ca   = P_N2_ca  / P_ca
x_H2O_ca  = P_H2O_ca / P_ca

M_mix_ca  = x_O2_ca * M_O2 + x_N2_ca * M_N2 + x_H2O_ca * M_H2O

cathode_Y_o2  = (x_O2_ca  * M_O2)  / M_mix_ca
cathode_Y_h2o = (x_H2O_ca * M_H2O) / M_mix_ca

# ============================================================
# 결과 출력
# ============================================================
print("=" * 45)
print(f"  [Anode]  T={T_an}°C, RH={RH_an}%, P={P_an} Pa")
print("=" * 45)
print(f"  P_sat(H2O)   = {P_sat_an:,.1f} Pa")
print(f"  P_H2O        = {P_H2O_an:,.1f} Pa")
print(f"  P_H2         = {P_H2_an:,.1f} Pa")
print(f"  anode_Y_h2   = {anode_Y_h2:.4f}")
print(f"  anode_Y_h2o  = {anode_Y_h2o:.4f}")
print()
print("=" * 45)
print(f"  [Cathode] T={T_ca}°C, RH={RH_ca}%, P={P_ca} Pa")
print("=" * 45)
print(f"  P_sat(H2O)    = {P_sat_ca:,.1f} Pa")
print(f"  P_H2O         = {P_H2O_ca:,.1f} Pa")
print(f"  P_O2          = {P_O2_ca:,.1f} Pa")
print(f"  cathode_Y_o2  = {cathode_Y_o2:.4f}")
print(f"  cathode_Y_h2o = {cathode_Y_h2o:.4f}")

## - Run Calculater

1. Standard initialization
2. hybrid initialization
3. species Patch
4. 단순 유동
5. 전기화학 Patch
6. 전기화학 반응 

In [ ]:
solver.settings.solution.initialization.standard_initialize()
solver.settings.solution.initialization.hybrid_initialize()

# 1. h2 patch (anode side)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["an-chn", "an-ctl", "an-gdl", "an-mpl"],
    variable="species-0",
    value=anode_Y_h2
)

# 2. o2 patch (cathode side)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["ca-chn", "ca-ctl", "ca-gdl", "ca-mpl", "membrn"],
    variable="species-1",
    value=cathode_Y_o2
)

# 3. h2o patch (anode side)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["an-chn", "an-ctl", "an-gdl", "an-mpl"],
    variable="species-2",
    value=anode_Y_h2o
)

# 4. h2o patch (cathode side)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["ca-chn", "ca-ctl", "ca-gdl", "ca-mpl", "membrn"],
    variable="species-2",
    value=cathode_Y_h2o
)

# 5. water content patch (UDS-3: w_lam)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["an-bpp", "an-chn", "an-ctl", "an-gdl", "an-mpl",
                "ca-bpp", "ca-chn", "ca-ctl", "ca-gdl", "ca-mpl", "membrn"],
    variable="uds-3",
    value=14
)

solver.settings.solution.run_calculation.iter_count = iter_init
solver.settings.solution.run_calculation.calculate()

# 1. Solid potential patch (anode side)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["an-bpp","an-chn", "an-ctl", "an-gdl", "an-mpl"],
    variable="uds-0",
    value=-0.03
)

# 2. Solid potential patch (cathode side)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["ca-bpp","ca-chn", "ca-ctl", "ca-gdl", "ca-mpl", "membrn"],
    variable="uds-0",
    value= oper_volt+0.03
)

# 3. Electrolyte potential patch (cathode side)
solver.settings.solution.initialization.patch.calculate_patch(
    domain="mixture",
    cell_zones=["ca-bpp","ca-chn", "ca-ctl", "ca-gdl", "ca-mpl", "membrn"],
    variable="uds-1",
    value=-0.15
)

solver.settings.solution.run_calculation.iter_count = iter_init
solver.settings.solution.run_calculation.calculate()


## - Parameter study
1. Boundary condition에서 ca-tab에 Solid potential value 지정
2. Iteration 지정
3. Calculate 진행
4. 수렴 후 case-data 파일 저장
- 이후 반복

In [ ]:
# IV parameter study 자동화
import os

for v in voltage_range:
    solver.settings.setup.boundary_conditions.wall = {
        "ca-tab": {'uds': {'uds': {'Solid potential': {'value': v}}}}
    }
    solver.settings.solution.run_calculation.iter_count = iter_second
    solver.settings.solution.run_calculation.calculate()
    
    out_path = os.path.join(output_dir, f"{int(v*100)}V.cas.h5")
    solver.file.write(file_type='case-data', file_name=out_path)

## - Mesh replacer
1. Mesh replace (기존 case 데이터 유지)
2. Mesh cell 개수 확인
3. Cell zone condition에 membrane material 설정 -> nafion-nr211로 변경

In [ ]:
# 기존 case에서 msh 파일 replace 명령
solver.file.replace_mesh(file_name=mesh_file)
solver.mesh.size_info()
solver.settings.setup.cell_zone_conditions.fluid = {
    'membrn': {'name': 'membrn', 'general': {'material': membrane_material}}
}